[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/material-candidate-explorer/blob/main/MATERIAL_APPLICATION_RECOMMENDER_T4.ipynb)

# Material Application Recommender — T4

Ask where a material should be used or which materials fit a component. The notebook covers the 12 code-owned material fields, decomposes broad questions into role-specific portfolios, optionally retrieves source-grounded RAG evidence, and returns candidate-family seeds or supplied candidates with score semantics, reasons, citations, uncertainty, and next validators.

This notebook never ranks unlike roles together. Retrieval seeds and literature citations do not receive material-performance credit. A scalar decision score appears only when you supply condition-complete named-validator observations and explicit weights. Missing evidence stays `UNKNOWN`.

The application decision run performs routing, retrieval, and evidence-closed ranking only; it never executes a generator, specialist property model, relaxation, DFT, or experiment. The notebook writes a per-role manual handoff receipt and never automatically chooses among unlike roles or promotes a retrieval seed into a structure. An optional later cell can execute the typed bulk-crystal bridge only after you explicitly select one condition-complete `bulk_crystal` role, supply immutable parent/goal/run-config JSON, configure MatterGen + MatterSim + CHGNet URLs, and opt in.

For configured 8–32 bulk-crystal generation with MatterGen, MatterSim, CHGNet, duplicate rejection, adaptive rounds, composition-scoped Pareto screening, scoped novelty lookup, and non-executed DFT input preparation, use [MATERIAL_CANDIDATE_DISCOVERY_T4.ipynb](https://colab.research.google.com/github/jujumelona/material-candidate-explorer/blob/main/MATERIAL_CANDIDATE_DISCOVERY_T4.ipynb). That notebook fails closed when required sidecars, credentials, unique candidates, or validator evidence are unavailable.

In [ ]:
# @title 0. Application question and evidence settings
APPLICATION_QUESTION = "Which materials fit the positive and negative electrodes of a fast-charge sodium-ion battery at room temperature?" # @param {type:"string"}
MATERIAL_FIELD = "AUTO" # @param ["AUTO", "general_inorganic", "battery_electrode", "solid_electrolyte", "superconductor", "heterogeneous_catalyst", "semiconductor", "photovoltaic_absorber", "thermoelectric", "magnetic_material", "ferroelectric_piezoelectric", "structural_alloy", "porous_framework"]
MAIN_MODEL_ROUTING = "AUTO" # @param ["AUTO", "REQUIRED", "OFF"]
REQUIRE_CONDITION_COMPLETE = False # @param {type:"boolean"}
INCLUDE_RETRIEVAL_SEEDS = True # @param {type:"boolean"}
RUN_RAG = False # @param {type:"boolean"}
APPLICATION_CONTEXT_JSON = "{}" # @param {type:"string"}
EXPLICIT_ROLE_IDS = "" # @param {type:"string"}
CHEMICAL_SYSTEM = "" # @param {type:"string"}

# Optional real Fusion execution. Keep OFF for broad or multi-role questions.
RUN_SINGLE_ROLE_BULK_SEARCH = False # @param {type:"boolean"}
BULK_SEARCH_ROLE_ID = "" # @param {type:"string"}
BULK_SEARCH_INPUT_MODE = "UPLOAD" # @param ["UPLOAD", "PATHS"]
BULK_SEARCH_GOAL_FILE = "goal.json" # @param {type:"string"}
BULK_SEARCH_PARENT_FILE = "parent-candidate.json" # @param {type:"string"}
BULK_SEARCH_RUN_CONFIG_FILE = "run-config.json" # @param {type:"string"}
BULK_SEARCH_ROUNDS = 4 # @param {type:"integer", min:3, max:12}
BULK_SEARCH_TOTAL_CANDIDATES = 16 # @param {type:"integer", min:8, max:32}
BULK_SEARCH_MAX_GENERATION_CALLS = 32 # @param {type:"integer", min:3, max:128}

# Optional strict JSON arrays. Leave [] to use only unscored retrieval seeds.
CANDIDATES_JSON = "[]" # @param {type:"string"}
OBSERVATIONS_JSON = "[]" # @param {type:"string"}
PREFERENCES_JSON = "[]" # @param {type:"string"}

# Optional source-grounded RAG/main-AI endpoint.
RAG_MODEL_API_URL = "" # @param {type:"string"}
RAG_MODEL_NAME = "" # @param {type:"string"}
RAG_MODEL_TIMEOUT_SECONDS = 180 # @param {type:"integer", min:1, max:1800}
MATERIAL_FIELD_MODEL_API_URL = "" # @param {type:"string"}
MATERIAL_FIELD_MODEL_NAME = "" # @param {type:"string"}
MATERIAL_FIELD_MODEL_TIMEOUT_SECONDS = 180 # @param {type:"integer", min:1, max:1800}
MATTERGEN_API_URL = "" # @param {type:"string"}
MATTERSIM_API_URL = "" # @param {type:"string"}
CHGNET_API_URL = "" # @param {type:"string"}
CONTACT_EMAIL = "" # @param {type:"string"}
RAG_FROM_DATE = "2020-01-01" # @param {type:"string"}
RAG_TO_DATE = "" # @param {type:"string"}
RAG_MAX_RESULTS = 12 # @param {type:"integer", min:1, max:50}

# Optional administrator-owned read-only MCP evidence tool.
MATERIAL_RAG_MCP_URL = "" # @param {type:"string"}
MATERIAL_APPLICATION_RAG_MCP_TOOL = "" # @param {type:"string"}
MCP_TOOL_GENERATION_PRIOR = "" # @param {type:"string"}
MCP_TOOL_IDENTITY_NOVELTY = "" # @param {type:"string"}
MCP_TOOL_MLIP_DISAGREEMENT = "" # @param {type:"string"}
MCP_TOOL_RELAXATION_VALIDATION = "" # @param {type:"string"}
MCP_TOOL_DFT_HANDOFF = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TIMEOUT_SECONDS = 60 # @param {type:"integer", min:1, max:1800}
MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP = False # @param {type:"boolean"}

PROJECT_REPOSITORY = "https://github.com/jujumelona/material-candidate-explorer.git"
PROJECT_REF = "main" # @param {type:"string"}
ARTIFACT_ROOT = "/content/material-application-results"

from datetime import date
import json
if not APPLICATION_QUESTION.strip():
    raise ValueError("APPLICATION_QUESTION is required.")
if MAIN_MODEL_ROUTING not in {"AUTO", "REQUIRED", "OFF"}:
    raise ValueError("MAIN_MODEL_ROUTING must be AUTO, REQUIRED, or OFF.")
if bool(RAG_MODEL_API_URL.strip()) != bool(RAG_MODEL_NAME.strip()):
    raise ValueError("RAG_MODEL_API_URL and RAG_MODEL_NAME must be set together.")
if bool(MATERIAL_FIELD_MODEL_API_URL.strip()) != bool(MATERIAL_FIELD_MODEL_NAME.strip()):
    raise ValueError("MATERIAL_FIELD_MODEL_API_URL and MATERIAL_FIELD_MODEL_NAME must be set together.")
if RUN_SINGLE_ROLE_BULK_SEARCH:
    if not BULK_SEARCH_ROLE_ID.strip():
        raise ValueError("BULK_SEARCH_ROLE_ID is required for explicit execution.")
    if BULK_SEARCH_INPUT_MODE not in {"UPLOAD", "PATHS"}:
        raise ValueError("BULK_SEARCH_INPUT_MODE must be UPLOAD or PATHS.")
    if not 3 <= BULK_SEARCH_ROUNDS <= 12:
        raise ValueError("BULK_SEARCH_ROUNDS must be between 3 and 12.")
    if not 8 <= BULK_SEARCH_TOTAL_CANDIDATES <= 32:
        raise ValueError("BULK_SEARCH_TOTAL_CANDIDATES must be between 8 and 32.")
    if not 3 <= BULK_SEARCH_MAX_GENERATION_CALLS <= 128:
        raise ValueError("BULK_SEARCH_MAX_GENERATION_CALLS must be between 3 and 128.")
    if not all(item.strip() for item in (
        BULK_SEARCH_GOAL_FILE,
        BULK_SEARCH_PARENT_FILE,
        BULK_SEARCH_RUN_CONFIG_FILE,
    )):
        raise ValueError("All three bulk-search JSON file names are required.")
    if not all(item.strip() for item in (
        MATTERGEN_API_URL,
        MATTERSIM_API_URL,
        CHGNET_API_URL,
    )):
        raise ValueError("MatterGen, MatterSim, and CHGNet API URLs are all required.")
MCP_TOOL_VALUES = [
    MATERIAL_APPLICATION_RAG_MCP_TOOL, MCP_TOOL_GENERATION_PRIOR,
    MCP_TOOL_IDENTITY_NOVELTY, MCP_TOOL_MLIP_DISAGREEMENT,
    MCP_TOOL_RELAXATION_VALIDATION, MCP_TOOL_DFT_HANDOFF,
]
if bool(MATERIAL_RAG_MCP_URL.strip()) != any(item.strip() for item in MCP_TOOL_VALUES):
    raise ValueError("Set MATERIAL_RAG_MCP_URL together with at least one application or stage MCP tool.")
if not 1 <= RAG_MAX_RESULTS <= 50:
    raise ValueError("RAG_MAX_RESULTS must be between 1 and 50.")
if not 1 <= RAG_MODEL_TIMEOUT_SECONDS <= 1800:
    raise ValueError("RAG_MODEL_TIMEOUT_SECONDS must be between 1 and 1800.")
if not 1 <= MATERIAL_FIELD_MODEL_TIMEOUT_SECONDS <= 1800:
    raise ValueError("MATERIAL_FIELD_MODEL_TIMEOUT_SECONDS must be between 1 and 1800.")
if not 1 <= MATERIAL_RAG_MCP_TIMEOUT_SECONDS <= 1800:
    raise ValueError("MATERIAL_RAG_MCP_TIMEOUT_SECONDS must be between 1 and 1800.")
rag_from_date = date.fromisoformat(RAG_FROM_DATE) if RAG_FROM_DATE.strip() else None
rag_to_date = date.fromisoformat(RAG_TO_DATE) if RAG_TO_DATE.strip() else None
if rag_from_date and rag_to_date and rag_from_date > rag_to_date:
    raise ValueError("RAG_FROM_DATE cannot be after RAG_TO_DATE.")
for label, raw in {
    "APPLICATION_CONTEXT_JSON": APPLICATION_CONTEXT_JSON,
    "CANDIDATES_JSON": CANDIDATES_JSON,
    "OBSERVATIONS_JSON": OBSERVATIONS_JSON,
    "PREFERENCES_JSON": PREFERENCES_JSON,
}.items():
    parsed = json.loads(raw)
    if label == "APPLICATION_CONTEXT_JSON" and not isinstance(parsed, dict):
        raise ValueError(label + " must be a JSON object.")
    if label != "APPLICATION_CONTEXT_JSON" and not isinstance(parsed, list):
        raise ValueError(label + " must be a JSON array.")
print("Settings validated. Broad questions will produce separate role portfolios.")

In [ ]:
# @title 1. Install the public project
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/content/material-candidate-explorer")
if not PROJECT_ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", PROJECT_REF,
        PROJECT_REPOSITORY, str(PROJECT_ROOT),
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)
], check=True)
print("Installed", PROJECT_ROOT)

In [ ]:
# @title 2. Route the question, optionally retrieve RAG evidence, and build portfolios
from datetime import date
from getpass import getpass
import os

from discovery_os.material_decision_runner import MaterialDecisionRunner
from discovery_os.material_recommendation import (
    MaterialApplicationCandidate,
    MaterialApplicationObservation,
    MaterialDecisionPreference,
)

def hidden_environment(name: str, prompt: str, enabled: bool) -> None:
    if not enabled:
        return
    value = getpass(prompt).strip()
    if value:
        os.environ[name] = value

if RAG_MODEL_API_URL.strip():
    os.environ["RAG_MODEL_API_URL"] = RAG_MODEL_API_URL.strip()
    os.environ["RAG_MODEL_NAME"] = RAG_MODEL_NAME.strip()
    os.environ["RAG_MODEL_TIMEOUT_SECONDS"] = str(RAG_MODEL_TIMEOUT_SECONDS)
    hidden_environment("RAG_MODEL_API_KEY", "RAG/main-model API key (optional, hidden): ", True)
if MATERIAL_FIELD_MODEL_API_URL.strip():
    os.environ["MATERIAL_FIELD_MODEL_API_URL"] = MATERIAL_FIELD_MODEL_API_URL.strip()
    os.environ["MATERIAL_FIELD_MODEL_NAME"] = MATERIAL_FIELD_MODEL_NAME.strip()
    os.environ["MATERIAL_FIELD_MODEL_TIMEOUT_SECONDS"] = str(MATERIAL_FIELD_MODEL_TIMEOUT_SECONDS)
    hidden_environment("MATERIAL_FIELD_MODEL_API_KEY", "Main material-routing API key (optional, hidden): ", True)
for name, value in {
    "MATTERGEN_API_URL": MATTERGEN_API_URL,
    "MATTERSIM_API_URL": MATTERSIM_API_URL,
    "CHGNET_API_URL": CHGNET_API_URL,
}.items():
    if value.strip():
        os.environ[name] = value.strip()
if RUN_SINGLE_ROLE_BULK_SEARCH:
    hidden_environment("MATTERGEN_API_TOKEN", "MatterGen API token (optional, hidden): ", True)
    hidden_environment("MATTERSIM_API_TOKEN", "MatterSim API token (optional, hidden): ", True)
    hidden_environment("CHGNET_API_TOKEN", "CHGNet API token (optional, hidden): ", True)
if CONTACT_EMAIL.strip():
    os.environ["LITERATURE_CONTACT_EMAIL"] = CONTACT_EMAIL.strip()
if RUN_RAG:
    hidden_environment("OPENALEX_API_KEY", "OpenAlex API key (hidden; press Enter to keep that source unavailable): ", True)
if MATERIAL_RAG_MCP_URL.strip():
    os.environ["MATERIAL_RAG_MCP_URL"] = MATERIAL_RAG_MCP_URL.strip()
    os.environ["MATERIAL_RAG_MCP_TIMEOUT_SECONDS"] = str(MATERIAL_RAG_MCP_TIMEOUT_SECONDS)
    os.environ["MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP"] = (
        "1" if MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP else "0"
    )
    for name, value in {
        "MATERIAL_APPLICATION_RAG_MCP_TOOL": MATERIAL_APPLICATION_RAG_MCP_TOOL,
        "MATERIAL_RAG_MCP_TOOL_GENERATION_PRIOR": MCP_TOOL_GENERATION_PRIOR,
        "MATERIAL_RAG_MCP_TOOL_IDENTITY_NOVELTY": MCP_TOOL_IDENTITY_NOVELTY,
        "MATERIAL_RAG_MCP_TOOL_MLIP_DISAGREEMENT": MCP_TOOL_MLIP_DISAGREEMENT,
        "MATERIAL_RAG_MCP_TOOL_RELAXATION_VALIDATION": MCP_TOOL_RELAXATION_VALIDATION,
        "MATERIAL_RAG_MCP_TOOL_DFT_HANDOFF": MCP_TOOL_DFT_HANDOFF,
    }.items():
        if value.strip():
            os.environ[name] = value.strip()
    hidden_environment("MATERIAL_RAG_MCP_TOKEN", "MCP bearer token (optional, hidden): ", True)

context = json.loads(APPLICATION_CONTEXT_JSON)
candidate_rows = [MaterialApplicationCandidate.model_validate(item, strict=True) for item in json.loads(CANDIDATES_JSON)]
observation_rows = [MaterialApplicationObservation.model_validate(item, strict=True) for item in json.loads(OBSERVATIONS_JSON)]
preference_rows = [MaterialDecisionPreference.model_validate(item, strict=True) for item in json.loads(PREFERENCES_JSON)]
role_ids = [item.strip() for item in EXPLICIT_ROLE_IDS.split(",") if item.strip()] or None

decision_run = MaterialDecisionRunner(artifact_root=ARTIFACT_ROOT).run(
    APPLICATION_QUESTION,
    material_field=MATERIAL_FIELD,
    chemical_system=CHEMICAL_SYSTEM.strip() or None,
    problem_context=context,
    main_model_routing=MAIN_MODEL_ROUTING.casefold(),
    explicit_role_ids=role_ids,
    require_condition_complete=REQUIRE_CONDITION_COMPLETE,
    include_retrieval_seeds=INCLUDE_RETRIEVAL_SEEDS,
    candidates=candidate_rows,
    observations=observation_rows,
    preferences=preference_rows,
    run_rag=RUN_RAG,
    # Leave sources unset so every stage applies its own allowlist and a
    # configured MCP tool is requested through the administrator-owned route.
    rag_from_date=rag_from_date,
    rag_to_date=rag_to_date,
    rag_max_results_per_query=RAG_MAX_RESULTS,
)
print(json.dumps({
    "run_id": decision_run.run_id,
    "field": decision_run.brief.material_field,
    "question_kind": decision_run.brief.question_kind,
    "roles": [item.role_id for item in decision_run.brief.roles],
    "cross_role_ranking": decision_run.report.cross_role_ranking_performed,
    "generation_or_specialist_execution_performed": decision_run.generation_or_specialist_execution_performed,
    "rag_bundle_id": decision_run.rag_bundle_id,
    "rag_stages": {
        item.evidence_stage: item.status
        for item in decision_run.rag_stage_receipts
    },
    "artifacts": [item.relative_path for item in decision_run.artifacts],
}, ensure_ascii=False, indent=2))

In [ ]:
# @title 3. Display candidates, score semantics, reasons, uncertainty, and next validation
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

rows = []
for portfolio in decision_run.report.role_recommendations:
    for item in portfolio.candidates:
        rows.append({
            "role": portfolio.role_id,
            "candidate": item.candidate.material_or_stack,
            "origin": item.candidate.origin,
            "condition_group": item.comparison_group_id or "UNKNOWN",
            "rank": item.rank_within_role_and_condition,
            "pareto_front": item.pareto_front,
            "hard_gates": item.hard_gate_status,
            "decision_score": item.pool_relative_decision_score,
            "score_semantics": item.score_semantics,
            "evidence_complete_%": item.evidence_completeness_score,
            "uncertainty": item.evidence_uncertainty_status,
            "model_disagreement": item.candidate.model_disagreement,
            "why": "; ".join(item.why_selected),
            "why_not_top": "; ".join(item.why_not_top),
            "tradeoffs": "; ".join(item.main_tradeoffs),
            "citations": "; ".join(c.doi or c.record_id for c in item.citations),
            "next_validation": item.next_validations[0] if item.next_validations else "",
        })
result_table = pd.DataFrame(rows)
display(result_table)
if decision_run.rag_stage_receipts:
    display(pd.DataFrame([
        {
            "evidence_stage": item.evidence_stage,
            "status": item.status,
            "mcp_tool": item.selected_mcp_tool or "UNCONFIGURED",
            "source_bundle": item.source_bundle_id or "NONE",
            "error": item.error or "",
        }
        for item in decision_run.rag_stage_receipts
    ]))

display(Markdown(
    "**Scientific boundary:** scores are role- and condition-local decision support, "
    "not probabilities. `UNKNOWN` is not zero or pass. Retrieval seeds need exact "
    "source closure and named-validator observations before scientific ranking."
))
if decision_run.report.unresolved_questions:
    display(Markdown("### Missing context\n" + "\n".join(
        "- " + item for item in decision_run.report.unresolved_questions
    )))

# Build an explicit manual handoff. This receipt never starts the crystal notebook.
role_handoffs = []
for role in decision_run.brief.roles:
    missing_context = decision_run.brief.missing_context_by_role[role.role_id]
    bulk_scope_available = "bulk_crystal" in role.representation_scopes
    if missing_context:
        handoff_status = "operator_context_required"
    elif not CHEMICAL_SYSTEM.strip():
        handoff_status = "operator_chemical_system_required"
    elif not bulk_scope_available:
        handoff_status = "not_compatible_with_bulk_crystal_t4"
    elif role.bulk_cif_scope == "insufficient-interface-or-device-required":
        handoff_status = "bulk_triage_only_interface_or_device_validation_required"
    else:
        handoff_status = "manual_bulk_crystal_triage_ready"
    t4_inputs = None
    if CHEMICAL_SYSTEM.strip() and bulk_scope_available:
        t4_inputs = {
            "DISCOVERY_PROMPT": (
                f"Screen bulk crystal candidates for role '{role.role_id}' under: "
                + APPLICATION_QUESTION.strip()
            ),
            "MATERIAL_FIELD": decision_run.brief.material_field.value,
            "CHEMICAL_SYSTEM": CHEMICAL_SYSTEM.strip(),
            "MATERIAL_PROBLEM_CONTEXT_JSON": json.dumps(
                decision_run.brief.target_context,
                ensure_ascii=False,
                sort_keys=True,
            ),
        }
    role_handoffs.append({
        "role_id": role.role_id,
        "display_name": role.display_name,
        "handoff_status": handoff_status,
        "representation_scopes": list(role.representation_scopes),
        "bulk_cif_scope": role.bulk_cif_scope,
        "missing_context": missing_context,
        "t4_input_values": t4_inputs,
        "application_claim_boundary": role.claim_boundary,
    })
downstream_handoff = {
    "schema_version": "1.0",
    "source_run_id": decision_run.run_id,
    "automatic_role_selection_performed": False,
    "automatic_transfer_or_execution_performed": False,
    "application_decision_run_generation_or_specialist_execution_performed": (
        decision_run.generation_or_specialist_execution_performed
    ),
    "operator_action": (
        "Choose exactly one role, resolve its missing context and chemical system, "
        "then copy only that role's t4_input_values into "
        "MATERIAL_CANDIDATE_DISCOVERY_T4.ipynb."
    ),
    "role_handoffs": role_handoffs,
}
run_root = Path(ARTIFACT_ROOT) / decision_run.run_id
handoff_path = run_root / "application-to-crystal-t4-handoff.json"
handoff_path.write_text(
    json.dumps(downstream_handoff, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
display(Markdown(
    "### Manual crystal-search handoff\n"
    "No generator or specialist validator ran here. Choose one compatible role; "
    "the receipt records exact T4 input values only when a chemical system was supplied."
))
display(pd.DataFrame([
    {
        "role": item["role_id"],
        "handoff_status": item["handoff_status"],
        "bulk_cif_scope": item["bulk_cif_scope"],
        "missing_context": ", ".join(item["missing_context"]),
    }
    for item in role_handoffs
]))
print("Saved manual handoff receipt:", handoff_path)

In [ ]:
# @title 4. Optionally execute one validated bulk-crystal role
import subprocess
import sys

bulk_search_result = None
if not RUN_SINGLE_ROLE_BULK_SEARCH:
    print("Explicit bulk-crystal Fusion search skipped.")
else:
    selected_roles = decision_run.brief.roles
    if len(selected_roles) != 1:
        raise ValueError(
            "Explicit execution requires exactly one selected role. Set "
            "EXPLICIT_ROLE_IDS to one role and rerun; broad portfolios remain manual."
        )
    selected_role = selected_roles[0]
    if selected_role.role_id != BULK_SEARCH_ROLE_ID.strip():
        raise ValueError("BULK_SEARCH_ROLE_ID does not match the one selected role.")
    if "bulk_crystal" not in selected_role.representation_scopes:
        raise ValueError("The selected role has no bulk_crystal execution scope.")
    if not decision_run.brief.ready_for_condition_complete_scoring:
        raise ValueError(
            "Resolve every required role condition before explicit Fusion execution."
        )
    if decision_run.brief.field_plan.resolution.requires_operator_choice:
        raise ValueError("Ambiguous material-field routing cannot execute a search.")

    required_input_names = [
        BULK_SEARCH_GOAL_FILE,
        BULK_SEARCH_PARENT_FILE,
        BULK_SEARCH_RUN_CONFIG_FILE,
    ]
    if BULK_SEARCH_INPUT_MODE == "UPLOAD":
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError(
                "UPLOAD mode requires Colab; use PATHS outside Colab."
            ) from exc
        uploaded = files.upload()
        missing_uploads = [name for name in required_input_names if name not in uploaded]
        if missing_uploads:
            raise ValueError("Missing uploaded files: " + ", ".join(missing_uploads))
    input_paths = [Path(name).resolve() for name in required_input_names]
    missing_paths = [str(path) for path in input_paths if not path.is_file()]
    if missing_paths:
        raise ValueError("Missing bulk-search JSON files: " + ", ".join(missing_paths))

    # Validate the exact typed inputs before any sidecar call. The bridge repeats
    # these checks and rejects wrong hashes, profiles, objectives, or structures.
    from discovery_os.fusion_schemas import WorkspaceRunConfig
    from discovery_os.schemas import Candidate, DiscoveryGoal
    goal_input = DiscoveryGoal.model_validate_json(
        input_paths[0].read_text(encoding="utf-8"), strict=True
    )
    parent_input = Candidate.model_validate_json(
        input_paths[1].read_text(encoding="utf-8"), strict=True
    )
    run_config_input = WorkspaceRunConfig.model_validate_json(
        input_paths[2].read_text(encoding="utf-8"), strict=True
    )
    if {item.property_name for item in goal_input.objectives} - {
        "energy_per_atom", "max_force"
    }:
        raise ValueError(
            "The bulk bridge accepts only energy_per_atom and max_force diagnostics."
        )
    if run_config_input.generator_id != "mattergen":
        raise ValueError("The explicit notebook bridge requires generator_id=mattergen.")
    if run_config_input.candidate_count > BULK_SEARCH_TOTAL_CANDIDATES:
        raise ValueError("Per-call candidate_count exceeds the global candidate budget.")

    search_id = f"{decision_run.run_id}-{selected_role.role_id}"
    search_artifact_root = run_root / "explicit-bulk-crystal-search"
    command = [
        sys.executable, "-m", "discovery_os", "material-fusion-search",
        "--brief", str(run_root / "application-brief.json"),
        "--role", selected_role.role_id,
        "--search-id", search_id,
        "--goal", str(input_paths[0]),
        "--parent", str(input_paths[1]),
        "--run-config", str(input_paths[2]),
        "--generator", "mattergen",
        "--rounds", str(BULK_SEARCH_ROUNDS),
        "--frontier-width", "1",
        "--no-control-sweep",
        "--max-generation-calls", str(BULK_SEARCH_MAX_GENERATION_CALLS),
        "--max-generated-candidates", str(BULK_SEARCH_TOTAL_CANDIDATES),
        "--expert", "mattersim",
        "--expert", "chgnet",
        "--required-evaluator", "mattersim",
        "--required-evaluator", "chgnet",
        "--artifacts", str(search_artifact_root),
    ]
    completed = subprocess.run(
        command,
        check=True,
        text=True,
        capture_output=True,
    )
    bulk_search_result = json.loads(completed.stdout)
    bridge_report_path = run_root / "explicit-bulk-crystal-search-report.json"
    bridge_report_path.write_text(
        json.dumps(bulk_search_result, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    downstream_handoff["explicit_opt_in_bulk_search"] = {
        "execution_performed": True,
        "role_id": selected_role.role_id,
        "search_id": search_id,
        "report": str(bridge_report_path.relative_to(run_root)),
        "application_property_scoring_performed": False,
        "application_claim_created": False,
    }
    handoff_path.write_text(
        json.dumps(downstream_handoff, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    search_report = bulk_search_result["report"]["search"]["report"]
    print(json.dumps({
        "execution": "completed",
        "role_id": selected_role.role_id,
        "search_id": search_id,
        "status": search_report["status"],
        "rounds_completed": search_report["rounds_completed"],
        "generated_candidates": search_report["budget_usage"]["generated_candidates"],
        "application_property_scoring_performed": False,
        "application_claim_created": False,
        "report": str(bridge_report_path),
    }, ensure_ascii=False, indent=2))

In [ ]:
# @title 5. Export run artifacts, handoff, and optional bridge report
import shutil

archive_base = Path("/content") / decision_run.run_id
archive_path = shutil.make_archive(str(archive_base), "zip", run_root)
print("Created", archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Download manually outside Colab:", archive_path)